# 講師用：H₄の計算結果を再生成する
受講者版では保存済み結果を使います。このノートは講師が専用環境で使用します。


In [ ]:
import sys, subprocess, importlib.util
from pathlib import Path
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
if IS_COLAB:
    subprocess.run([sys.executable,"-m","pip","install","-q","qiskit==2.4.1"],check=True)
    checkout=Path("/content/IBM_handson")
    if not checkout.exists():
        subprocess.run(["git","clone","https://github.com/QS-laboratory/IBM_handson.git",str(checkout)],check=True)
    EXERCISE_ROOT=checkout/"advanced_kanno_h4"
else:
    candidates=[Path.cwd(),Path.cwd()/"advanced_kanno_h4",Path.cwd().parent]
    EXERCISE_ROOT=next(p for p in candidates if (p/"src/author_analysis.py").exists())
if not EXERCISE_ROOT.exists():
    raise FileNotFoundError("発展演習フォルダをGitHubへ配置してからColabで開いてください")
sys.path.insert(0,str(EXERCISE_ROOT/"src"))
import json
import numpy as np
import matplotlib.pyplot as plt
from author_analysis import *
print("Exercise:",EXERCISE_ROOT)
print("No IBM jobs will be submitted.")

In [ ]:
GEOMETRY = 0
NAMES = ["canonical", "boys", "boys_ordered"]


## 5. 自分で再生成する（講師・上級者向け）

重い最適化は通常の本編時間に含めません。Colabでは配布結果の比較を標準とします。
全生成にはJuliaとITensorsが必要です。`README.md`の手順で専用環境を準備してください。
このセルは既定では再生成しません。実行するとこの独立フォルダ内の結果だけを更新します。

In [ ]:
REGENERATE_HAMILTONIANS=False
REGENERATE_TENSORS=False
REGENERATE_QPDE=False
if REGENERATE_HAMILTONIANS:
    subprocess.run([sys.executable,"-m","pip","install","-r",str(EXERCISE_ROOT/"requirements.txt")],check=True)
    subprocess.run([sys.executable,str(EXERCISE_ROOT/"src/h4_models.py"),"0","1","2","3","4"],check=True)
if REGENERATE_TENSORS:
    import shutil
    if shutil.which("julia") is None:
        raise RuntimeError("Juliaが必要です。READMEの再生成手順を参照してください")
    subprocess.run(["bash",str(EXERCISE_ROOT/"tools/setup_julia.sh")],check=True)
    subprocess.run(["bash",str(EXERCISE_ROOT/"tools/run_author.sh")],check=True)
if REGENERATE_QPDE:
    for name in NAMES:
        result=run_qpde(name,GEOMETRY,exact_probabilities=False)
        print(name,result["status"])
print("Regeneration flags:",REGENERATE_HAMILTONIANS,REGENERATE_TENSORS,REGENERATE_QPDE)